# Continuous-Time Mechanisms as Local Vector Fields on an OCP

**Author:** Nadine Mohamed  
**Date:** 2026-10-03

> Working research note. This notebook formalizes a view of mechanisms as local, state-dependent transformations of an optimal-control problem (OCP), with adaptive agents learning policies that select the parameters of those mechanisms.

Copyright © 2026 Nadine Mohamed. All rights reserved unless otherwise stated.

## 1. OCP as a time-dependent object

Let the optimal-control problem at time \(t\) be represented abstractly as an object

\[
p_t \in \mathcal P.
\]

The object \(p_t\) may collect the components that define the OCP, for example

\[
p_t =
\left(
\mathcal U_t,\,
\mathcal X_t,\,
\mathcal O_t,\,
I_t,\,
\ell_t,\,
T_t,\,
\ldots
\right).
\]

Some components are naturally numerical or function-valued and admit additive infinitesimal changes. Others, such as admissible state or action sets, are naturally transformed through set operations such as intersection.

For the continuously varying components, assume that the relevant part of \(\mathcal P\) has enough differentiable structure to speak about a tangent space \(T_{p_t}\mathcal P\).

## 2. A mechanism as a local vector-field contribution

A mechanism \(G_i\) specifies how the decision made by agent \(i\) changes the OCP locally and instantaneously.

For a mechanism parameterized by \(\theta_{i,t}\),

\[
G_i(p_t;\theta_{i,t}) \in T_{p_t}\mathcal P,
\]

and its contribution to the instantaneous evolution of the OCP is

\[
\boxed{
\dot p_t^{(i)}
=
G_i(p_t;\theta_{i,t}).
}
\]

The important distinction is:

- \(G_i\) defines the **structural effect** of the mechanism;
- \(\theta_{i,t}\) specifies the **particular mechanism setting chosen at time \(t\)**.

Thus \(G_i\) describes *what a given mechanism choice does to the OCP*.

## 3. The policy selects mechanism parameters

Agent \(i\) does not directly choose \(\dot p_t^{(i)}\). Instead, it chooses the mechanism parameters

\[
\boxed{
\theta_{i,t}
=
\pi_{\phi_i}(I_{i,0:t}),
}
\]

where \(I_{i,0:t}\) denotes the information available to the agent up to time \(t\), and \(\pi_{\phi_i}\) is its learned decision policy.

Substituting the policy into the mechanism gives

\[
\boxed{
\dot p_t^{(i)}
=
G_i\!\left(
p_t;
\pi_{\phi_i}(I_{i,0:t})
\right).
}
\]

The conceptual chain is therefore

\[
I_{i,0:t}
\longrightarrow
\pi_{\phi_i}
\longrightarrow
\theta_{i,t}
\longrightarrow
G_i
\longrightarrow
\dot p_t^{(i)}.
\]

The policy decides **which mechanism parameters to instantiate**; the mechanism specifies **how those parameters deform the OCP**.

## 4. Internal decomposition of the learned agent

A useful decomposition of the agent is

\[
h_{\psi_i}(I_{i,0:t}),
\]

a learned representation of the information/history available to agent \(i\);

\[
S_{\omega_i}
\left(
\pi_{-i}
\mid
h_{\psi_i}(I_{i,0:t})
\right),
\]

a model or belief about the strategies of the other agents; and

\[
\pi_{\eta_i},
\]

the decision rule mapping these internal representations to the mechanism parameters.

Thus

\[
\boxed{
\theta_{i,t}
=
\pi_{\eta_i}
\left(
h_{\psi_i}(I_{i,0:t}),
S_{\omega_i}
\left(
\pi_{-i}
\mid
h_{\psi_i}(I_{i,0:t})
\right)
\right).
}
\]

The full policy parameter set may be collected as

\[
\boxed{
\phi_i=(\psi_i,\omega_i,\eta_i).
}
\]

Here \(\psi_i,\omega_i,\eta_i\) should be understood as **parameter blocks**. In a neural implementation, each block may contain many weight matrices rather than a single matrix.

## 5. Multi-agent composition

Let \(G_0(p_t)\) denote the baseline or uncontrolled evolution of the OCP. If the mechanism-induced changes are additive, then \(N\) agents generate the total local OCP dynamics

\[
\boxed{
\dot p_t
=
G_0(p_t)
+
\sum_{i=1}^{N}
G_i
\left(
p_t;
\pi_{\phi_i}(I_{i,0:t})
\right).
}
\]

Equivalently,

\[
\dot p_t
=
G_0(p_t)
+
\sum_{i=1}^{N}
G_i(p_t;\theta_{i,t}),
\qquad
\theta_{i,t}
=
\pi_{\phi_i}(I_{i,0:t}).
\]

This yields a composable architecture: each agent contributes a local vector to the instantaneous OCP evolution, and the additive contributions can be evaluated independently and reduced through summation.

## 6. From instantaneous change to finite change

A vector field specifies an instantaneous rate of change. Over a sufficiently small interval \(\Delta t\),

\[
p(t+\Delta t)
\approx
p(t)
+
\dot p_t\,\Delta t.
\]

Substituting the multi-agent dynamics,

\[
p(t+\Delta t)
\approx
p(t)
+
\left[
G_0(p_t)
+
\sum_{i=1}^{N}
G_i(p_t;\theta_{i,t})
\right]\Delta t.
\]

Over a finite horizon, the continuous components accumulate these local changes through integration:

\[
\boxed{
p(T)
=
p(0)
+
\int_0^T
\left[
G_0(p_t)
+
\sum_{i=1}^{N}
G_i(p_t;\theta_{i,t})
\right]dt,
}
\]

whenever the chosen representation of \(p\) permits this additive integral form.

## 7. Objective and reward interpretation

Suppose a mechanism acts on an actor's accumulated objective \(J_j\). A regulator can choose parameters such as a quota, punishment intensity, subsidy, or monitoring intensity:

\[
\theta_{R,t}
=
\begin{bmatrix}
q_t\\
\gamma_t\\
\beta_t\\
\rho_t
\end{bmatrix}
=
\pi_{\phi_R}(I_{R,0:t}).
\]

The regulator mechanism then determines an instantaneous contribution to the actor's objective accumulation:

\[
\dot J_j^{(R)}
=
G_R^{J_j}(p_t;\theta_{R,t}).
\]

For several additive reward mechanisms,

\[
\boxed{
\dot J_j
=
r_j^0(p_t)
+
\sum_k
g_k(p_t;\theta_{k,t}).
}
\]

Thus incentives, penalties, and other reward-shaping mechanisms may be represented as composable contributions to the instantaneous rate of accumulation of the actor's objective.

## 8. Hard constraints are not ordinary additive vector fields

The vector-field formulation is most direct for continuously varying numerical or function-valued OCP components.

A hard quota or admissibility constraint may instead act through a set-valued transformation such as

\[
\mathcal U_t'
=
\mathcal U_t
\cap
\mathcal U_t^{(1)}
\cap \cdots \cap
\mathcal U_t^{(m)}.
\]

Likewise,

\[
\mathcal X_t'
=
\mathcal X_t
\cap
\mathcal X_t^{(1)}
\cap \cdots \cap
\mathcal X_t^{(m)}.
\]

These are naturally composed by **intersection**, not by ordinary vector addition.

A general mechanism framework can therefore be viewed as a **typed composition system**:

\[
\text{continuous OCP components}
\quad\longrightarrow\quad
\text{vector-field addition},
\]

\[
\text{admissible-set components}
\quad\longrightarrow\quad
\text{set intersection or other constraint operators}.
\]

If one wants a fully differential treatment of changing feasible sets, tangent cones or differential inclusions may be more appropriate than an ordinary smooth vector field.

## 9. Physical-space formulation

The same framework extends naturally when agents occupy a physical space.

Let agent \(i\) have spatial position

\[
z_i(t)\in\Omega\subseteq\mathbb R^d.
\]

The effect of agent \(i\)'s mechanism on another agent \(j\), or on an OCP component localized near \(z_j(t)\), may depend on spatial separation

\[
d_{ij}(t)
=
\|z_i(t)-z_j(t)\|.
\]

Introduce an interaction kernel

\[
K(d)\ge 0,
\]

for example

\[
K(d)=e^{-\lambda d},
\qquad \lambda>0.
\]

Then a spatially weighted mechanism contribution can be written

\[
\boxed{
G_{i\to j}
=
K(d_{ij}(t))
\,
G_i(p_t;\theta_{i,t}).
}
\]

The local dynamics experienced by agent \(j\) become

\[
\boxed{
\dot p_t^{(j)}
=
G_0^{(j)}(p_t)
+
\sum_{i\neq j}
K\!\left(
\|z_i(t)-z_j(t)\|
\right)
G_{i\to j}(p_t;\theta_{i,t}).
}
\]

A decaying kernel makes nearby agents exert stronger local influence than distant agents.

The notion of distance need not be Euclidean. The same construction can use:

- physical distance;
- graph distance;
- communication distance;
- social-network distance;
- institutional or jurisdictional distance;
- similarity in a learned embedding.

Hence the mechanism field can be **spatially or relationally local**.

## 10. Joint physical and OCP dynamics

If physical positions themselves evolve, one may define a joint state

\[
q_t=(p_t,z_1(t),\ldots,z_N(t)).
\]

Then the coupled system can contain both OCP evolution and physical motion:

\[
\dot p_t
=
G_0(p_t)
+
\sum_i
G_i
\left(
p_t,z_1,\ldots,z_N;
\theta_{i,t}
\right),
\]

\[
\dot z_i(t)
=
F_i
\left(
z_i(t),p_t,\theta_{i,t}
\right).
\]

This creates a coupled dynamical system in which agents may simultaneously:

1. move through physical or relational space;
2. choose mechanism parameters;
3. modify the OCP locally;
4. experience the modified OCP;
5. update their strategies from the resulting trajectories.

## 11. Interpretation

The framework separates four objects:

\[
\boxed{
\text{information representation}
\;\to\;
\text{strategic model}
\;\to\;
\text{policy}
\;\to\;
\text{mechanism field}.
}
\]

More explicitly,

\[
I_{i,0:t}
\overset{h_{\psi_i}}{\longrightarrow}
h_{i,t}
\overset{S_{\omega_i}}{\longrightarrow}
\text{belief/model of }\pi_{-i}
\overset{\pi_{\eta_i}}{\longrightarrow}
\theta_{i,t}
\overset{G_i}{\longrightarrow}
\dot p_t^{(i)}.
\]

The resulting view is:

> **An adaptive agent learns a policy that selects mechanism parameters; those parameters instantiate a local vector-field contribution describing the agent's instantaneous effect on the OCP.**

For additive mechanisms, several agents' contributions compose as a sum of local vector fields. For constraint-valued mechanisms, composition uses the corresponding set or constraint operator instead.

## 12. Research implications

This formulation suggests several distinct learning problems:

### Policy learning
Learn

\[
\pi_{\phi_i}: I_{i,0:t}\mapsto\theta_{i,t},
\]

so that the agent selects useful mechanism parameters.

### Opponent / strategy modeling
Learn or infer

\[
S_{\omega_i}(\pi_{-i}\mid h_{i,t}),
\]

or potentially a distribution over other agents' policy functions.

### Mechanism learning
If the effect law itself is uncertain, learn a parameterized mechanism family

\[
G_{i,\xi_i}(p_t;\theta_{i,t}),
\]

where \(\xi_i\) parameterizes the structural map from chosen mechanism settings to OCP change.

### Spatial mechanism learning
Learn either the interaction kernel

\[
K_{\kappa_i}(d)
\]

or a more general spatial field

\[
G_i(p_t,z;\theta_{i,t}),
\]

allowing the strength and geometry of institutional influence to adapt to location or network structure.

## 13. Compact formal statement

For each agent \(i\),

\[
\phi_i=(\psi_i,\omega_i,\eta_i),
\]

\[
\theta_{i,t}
=
\pi_{\eta_i}
\left(
h_{\psi_i}(I_{i,0:t}),
S_{\omega_i}
\left(
\pi_{-i}
\mid
h_{\psi_i}(I_{i,0:t})
\right)
\right),
\]

and

\[
\dot p_t^{(i)}
=
G_i(p_t;\theta_{i,t}).
\]

For additive continuous components of the OCP,

\[
\boxed{
\dot p_t
=
G_0(p_t)
+
\sum_{i=1}^{N}
G_i(p_t;\theta_{i,t}).
}
\]

With spatially local interactions,

\[
\boxed{
\dot p_t^{(j)}
=
G_0^{(j)}(p_t)
+
\sum_{i\neq j}
K(d_{ij}(t))
G_{i\to j}(p_t;\theta_{i,t}).
}
\]

This separates **decision learning** from **mechanism structure**, while retaining composability across agents and mechanisms.